# DL Masterclass 02: Activation Functions & Gradient Dynamics

In this module, we move beyond simple linear transformations to the heart of neural network depth: **Non-Linearity**. 

Without an activation function, the composition of linear layers $f(g(x))$ where $f(x) = W_2x$ and $g(x) = W_1x$ is just $W_2(W_1x) = (W_2W_1)x$, which is still linear. Activation functions are what allow neural networks to approximate any continuous function (Universal Approximation Theorem).

---

## Table of Contents
1. **Sigmoid, Tanh, and the Vanishing Gradient:** The mathematical decay of early AI.
2. **ReLU & The Subgradient Problem:** Sparse representations and the hardware king.
3. **The Dead Neuron Epidemic:** Diagnosis and treatment of necrotic networks.
4. **The Full Activation Zoo:** SiLU/Swish, Mish, ELU, and modern non-monotonics.
5. **Why GLU Variants Outperform:** The secret behind Llama 2/3 performance.
6. **Modern Era: Transformers & GELU:** Timing benchmarks and hardware optimizations.

---

## 🎓 Socratic Deep Check
**The Paradox of Choice:**
> *"If a 'perfect' activation function has a gradient of 1.0 everywhere (to prevent vanishing gradients) and is linear (for easy optimization), it becomes a linear identity function $f(x)=x$. But if it is non-linear to allow complex learning, it must have regions where the gradient is NOT 1.0. How do we design a function that is 'non-linear enough' to learn but 'linear enough' to not kill gradients?"*

---
📈 **Production Signal:** OpenAI's GPT-4 and Meta's Llama 3 use **SwiGLU** (a gated variant of SiLU). Understanding the transition from ReLU to SwiGLU is the difference between training 1M parameter models and 1T parameter models.

---


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** We select activation functions based on the **Gradient Flow Landscape**. Sigmoid/Tanh are used essentially only in output layers or gated recurrent units (LSTMs) because their "saturation regions" (where the function flattens) kill gradients. Modern architectures prioritize "Non-Saturating" activations like ReLU or "Smoothly Saturating" ones like SiLU/GELU.

**Mental Model:** Imagine pouring water (gradients) through a series of pipes (layers). 
- **Sigmoid** is a pipe with a filter that only lets 25% of the water through. After 10 pipes, you have almost nothing left.
- **ReLU** is a one-way valve: 100% flow for positive pressure, 0% flow for negative.
- **SwiGLU** is a smart valve that adjusts its flow based on the contents of a second parallel pipe.

**Common Junior Pitfall:** Thinking that SiLU or GELU is just a "better ReLU." At small scales, the difference is negligible. At large scales (LLMs), the smoothness of the derivative at $x=0$ prevents "Optimization Spikes" that can crash a multi-million dollar training run.

---


## 1. Sigmoid, Tanh, and the Vanishing Gradient

The Sigmoid function $\sigma(x) = \frac{1}{1 + e^{-x}}$ was the biological inspiration for early AI. It mimics a neuron's firing rate. However, it is the primary culprit of the **Vanishing Gradient Problem**.

### The Mathematical Decay
Let's derive the derivative of Sigmoid to see the bottleneck:
1. $\sigma(x) = (1 + e^{-x})^{-1}$
2. Using Power Rule + Chain Rule: $\sigma'(x) = -(1+e^{-x})^{-2} \cdot (-e^{-x})$
3. $\sigma'(x) = \frac{e^{-x}}{(1+e^{-x})^2}$
4. Simple Algebra: $\sigma'(x) = \frac{1}{1+e^{-x}} \cdot \frac{e^{-x}}{1+e^{-x}} = \sigma(x)(1 - \sigma(x))$

**The Bottleneck:** The maximum value of $\sigma(x)(1-\sigma(x))$ is exactly **0.25** (at $x=0$). 

In a 20-layer network, the gradient at the first layer involves: $\prod_{i=1}^{20} f'(z_i)$. 
Even in the best-case scenario (all $z=0$), the signal is multiplied by $0.25^{20} \approx 9 	imes 10^{-13}$. The weights in early layers receive almost zero update. They are "frozen" in their random initialization.

📈 **Production Signal:** Google's original PageRank algorithms relied on linear algebra, but their early deep learning for Search had to abandon Sigmoid for ReLU to move beyond 3-4 layers of depth.

### 🎓 Socratic Deep Check
> *"If Tanh has a maximum derivative of 1.0 (at $x=0$), which is 4x better than Sigmoid, why does it still suffer from vanishing gradients in very deep networks compared to ReLU?"*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

def sigmoid(x): return 1 / (1 + np.exp(-x))
def d_sigmoid(x): return sigmoid(x) * (1 - sigmoid(x))

def tanh(x): return np.tanh(x)
def d_tanh(x): return 1 - np.tanh(x)**2

x = np.linspace(-5, 5, 200)

plt.figure(figsize=(12, 5))
plt.plot(x, d_sigmoid(x), label="Sigmoid Derivative (Max 0.25)", lw=2)
plt.plot(x, d_tanh(x), label="Tanh Derivative (Max 1.0)", lw=2)
plt.axhline(0.1, color='red', linestyle='--', alpha=0.3, label="The 'Vanishing' Zone")
plt.title("The Gradient Bottleneck: Sigmoid vs Tanh")
plt.legend()
plt.show()

"""
What just happened?
We plotted the derivatives (slopes) of Sigmoid and Tanh. Notice how narrow the 'learning window' is.
If the input 'x' is greater than 3 or less than -3, the gradient effectively drops to near-zero.
This is called 'Saturation' - the neuron is finished learning because its slope is flat.
"""

## 2. ReLU & The Subgradient Problem

ReLU (Rectified Linear Unit) $\text{ReLU}(x) = \max(0, x)$ changed everything. 
Its derivative is exactly $1.0$ for $x > 0$. Through the chain rule, $1.0 \times 1.0 \times 1.0 = 1.0$, allowing gradients to flow through hundreds of layers without decaying.

### The Non-Differentiable Point
At $x=0$, the limit from the left (0) != the limit from the right (1). The derivative is undefined. 
In production frameworks (PyTorch/TensorFlow), we use **Subgradients**: we simply define $f'(0) = 0$ (or sometimes 0.5, though 0 is standard).

📈 **Production Signal:** ReLU is the default activation for almost all Convolutional Neural Networks (ResNets, EfficientNets) used in production computer vision today.


## 3. The Dead Neuron Epidemic — Diagnosis and Treatment

While ReLU solves vanishing gradients, it introduces a new pathology: **Dead Neurons**. 
If a weight update (gradient $\times$ learning rate) pushes a ReLU neuron's weights such that it outputs $<0$ for the entire dataset, that neuron's gradient becomes permanently 0. It will never fire again. It is "dead."

💡 **Senior Warning:** High learning rates + ReLU + poor initialization (leaving weights too large/negative) can cause up to 40% neuron death in industrial models, silently degrading model capacity without causing an explicit error.

### 🎓 Socratic Deep Check
> *"If a neuron is dead because its output is always negative, and its gradient is thus always zero, can it ever 'come back to life' using standard Gradient Descent? Why or why not?"*

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_moons

# 1. Create a dataset
X, y = make_moons(n_samples=1000, noise=0.1, random_state=42)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

def train_and_check_death(activation_fn, lr=10.0):
    model = nn.Sequential(
        nn.Linear(2, 50),
        activation_fn,
        nn.Linear(50, 50),
        activation_fn,
        nn.Linear(50, 2)
    )
    
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    # Train for 100 steps with an aggressively high LR to induce death
    for _ in range(100):
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
    # Diagnosis: Check how many neurons in the first hidden layer output 0 for ALL samples
    with torch.no_grad():
        hidden1 = activation_fn(model[0](X))
        # A neuron is 'dead' if its output is 0 for every sample in the batch
        dead_count = (hidden1.abs().sum(dim=0) == 0).sum().item()
        
    return dead_count / 50 * 100

relu_death = train_and_check_death(nn.ReLU())
leaky_death = train_and_check_death(nn.LeakyReLU(0.01))
elu_death = train_and_check_death(nn.ELU())

plt.bar(['ReLU', 'LeakyReLU', 'ELU'], [relu_death, leaky_death, elu_death], color=['red', 'blue', 'green'])
plt.ylabel("Death Rate (% of Neurons)")
plt.title("The Dead Neuron Epidemic (LR=10.0)")
plt.show()

"""
What just happened?
We deliberately sabotaged the training with a massive learning rate (10.0).
Standard ReLU shows a high percentage of dead neurons (often 20-50%).
LeakyReLU and ELU show 0% death because they always allow a small gradient to flow for negative values,
enabling the neuron to eventually 'learn' its way back to a positive output if needed.
"""

## 4. The Full Activation Function Zoo with Derivatives

As we moved into the Transformer era, researchers found that ReLU's "hard angle" at $x=0$ was too brutal for ultra-deep models. We need smoother approximations.

| Function | Formula | Derivative $f'(x)$ | Production Signal |
| :--- | :--- | :--- | :--- |
| **SiLU (Swish)** | $x \cdot \sigma(x)$ | $\sigma(x)(1 + x(1-\sigma(x)))$ | Default in **LLaMA 2/3** FFN layer. |
| **Mish** | $x \cdot \tanh(\text{softplus}(x))$ | Complex but smooth | Popular in **YOLOv4/v5** object detectors. |
| **ELU** | $\alpha(e^x - 1)$ for $x < 0$ | $f(x) + \alpha$ for $x < 0$ | Used in early high-accuracy CNNs to prevent bias shift. |
| **GELU** | $x \cdot \Phi(x)$ | $\Phi(x) + x \cdot \phi(x)$ | The standard in **BERT, GPT-2, ViT**. |

**Key Insight:** Why do these work better? They are all **Non-Monotonic** (they dip slightly below zero and then come back up). This allows a small amount of negative information to flow, which provides better regularization than ReLU's hard cutoff.

### 🎓 Socratic Deep Check
> *"Look at the SiLU formula $x \cdot \sigma(x)$. As $x \to \infty$, what does this function behave like? As $x \to -\infty$, what does it behave like? Why is this hybrid behavior desirable?"*

In [ ]:
def silu(x): return x * sigmoid(x)
def softplus(x): return np.log(1 + np.exp(x))
def mish(x): return x * np.tanh(softplus(x))
def elu(x, alpha=1.0): return np.where(x > 0, x, alpha * (np.exp(x) - 1))

def d_silu(x): 
    s = sigmoid(x)
    return s + x * s * (1 - s)

x_range = np.linspace(-4, 4, 300)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))

# Left: Activation Values
ax[0].plot(x_range, relu(x_range), label="ReLU", linestyle='--')
ax[0].plot(x_range, silu(x_range), label="SiLU (Swish)")
ax[0].plot(x_range, mish(x_range), label="Mish")
ax[0].plot(x_range, elu(x_range), label="ELU")
ax[0].set_title("Activation Functions")
ax[0].legend()

# Right: Derivatives
ax[1].plot(x_range, (x_range > 0).astype(float), label="ReLU", linestyle='--')
ax[1].plot(x_range, d_silu(x_range), label="SiLU Derivative")
ax[1].set_title("Derivative Comparison (Smoothness)")
ax[1].legend()

plt.show()

"""
What just happened?
We compared modern activations. Notice that SiLU/Mish/ELU are 'smooth' at zero,
unlike ReLU's sharp corner. More importantly, notice the 'dip' below zero
between x=-2 and x=0. This allows gradients to flow even for slightly negative values.
"""

## 5. Why GLU Variants Outperform Standard Activations in Transformers

In modern LLMs (Llama 2, Mistral, PaLM), we don't just use $f(Wx + b)$. We use **Gated Linear Units (GLU)**.

A GLU is a layer where one path acts as a "gate" for the other:
$$\text{GLU}(x, W, V, b, c) = \sigma(xW + b) \otimes (xV + c)$$

Here, $\otimes$ is element-wise multiplication. 
- Path 1 (Gate): $\sigma(xW + b)$ learns **which** information is relevant.
- Path 2 (Value): $(xV + c)$ is the raw content.

### The SwiGLU Evolution
Noam Shazeer (2020) proposed **SwiGLU**, which replaces Sigmoid with SiLU and removes biases:
$$\text{SwiGLU}(x, W, V) = \text{SiLU}(xW) \otimes xV$$

Research showed that SwiGLU improves perplexity by ~0.5 bits/token on language modeling — which at the scale of Trillion-parameter models is an astronomical gain compared to adding more layers.

📈 **Production Signal:** SwiGLU is the state-of-the-art standard for Feed-Forward Networks (FFN) in almost all Llama-based architectures.

### 🎓 Socratic Deep Check
> *"GLU variants have twice as many parameters ($W$ and $V$) as a standard ReLU layer. If we compared a ReLU layer with 2x more hidden units against a GLU layer with 1x hidden units (making them parameter-equivalent), GLU still wins. Why is 'gating' more expressive than just 'more neurons'?"*

In [ ]:
import torch.nn.functional as F

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        # The SwiGLU Core: SiLU(xW) * xV
        gate = self.w1(x)
        value = self.w2(x)
        # SwiGLU component
        intermediate = F.silu(gate) * value
        return self.w3(intermediate)

# Test the block
x_input = torch.randn(1, 10, 512) # (Batch, Seq, Dim)
model = SwiGLU(512, 2048)
output = model(x_input)
print(f"Input shape: {x_input.shape} -> Output shape: {output.shape}")

"""
What just happened?
We implemented the SwiGLU block used in Llama 3.
Notice how it splits the input into two paths (w1 and w2), transforms one with SiLU,
multiplies them, and then projects back with w3.
This 'learned gating' allows the model to suppress noise much more effectively than ReLU.
"""

## 6. Modern Era: Transformers & GELU

In BERT and early GPT models, **GELU (Gaussian Error Linear Unit)** was the standard. 
$$\text{GELU}(x) = x \cdot \Phi(x)$$
where $\Phi(x)$ is the Standard Normal CDF.

### Exact vs. Approximation
In the early days, calculating $\Phi(x)$ was computationally expensive on many GPUs. Researchers used a fast Tanh approximation:
$$\text{FastGELU}(x) = 0.5x(1 + \tanh[\sqrt{2/\pi}(x + 0.044715x^3)])$$

**Senior Clarification:** In modern PyTorch (`F.gelu()`), if you specify `approximate='none'` (default), it uses the exact formula because modern CUDA kernels have highly optimized Error Function (`erf`) implementations. The approximation is now mostly used for hardware compatibility (e.g., exporting to specialized AI edge chips or ONNX).

📈 **Production Signal:** While LLMs are moving to SwiGLU, **Vision Transformers (ViT)** and **BERT** derivatives still primarily use GELU.


In [ ]:
import time

def benchmark_activations():
    data = torch.randn(1000, 1000).cuda() if torch.cuda.is_available() else torch.randn(1000, 1000)
    
    activations = {
        "ReLU": F.relu,
        "GELU (Exact)": lambda x: F.gelu(x, approximate='none'),
        "GELU (Tanh)": lambda x: F.gelu(x, approximate='tanh'),
        "SiLU (Swish)": F.silu
    }
    
    results = {}
    for name, fn in activations.items():
        # Warmup
        for _ in range(10): fn(data)
        
        start = time.perf_counter()
        for _ in range(1000): 
            _ = fn(data)
        end = time.perf_counter()
        results[name] = (end - start) * 1000 # ms
        
    return results

perf = benchmark_activations()
for name, t in perf.items():
    print(f"{name:15}: {t:.4f} ms per 1000 calls")

"""
What just happened?
We timed the raw speed of activation functions on 1M values.
ReLU is almost always the fastest because it is a simple bit-gate (if x > 0).
GELU and SiLU require transcendental math (exp, erf, tanh). 
While GELU is ~2x slower than ReLU, in an LLM with 100B params, the 'overhead' of the activation
function is tiny compared to the massive matrix multiplications, making the 'better learning' of GELU well worth the cost.
"""

---
## ✅ Knowledge Check

### Q1: The Vanishing Gradient
Why does the product $0.25 \times 0.25 \times \dots$ happen in backpropagation for Sigmoid networks?
<details><summary>👉 Answer</summary>
The Chain Rule. The gradient of the loss with respect to early layer weights is the product of all subsequent layer derivatives. Since Sigmoid's max derivative is 0.25, the product shrinks exponentially toward zero with setiap extra layer.
</details>

### Q2: Sparsity vs. Smoothness
Why would you choose SiLU over ReLU in a 70-billion parameter transformer?
<details><summary>👉 Answer</summary>
In massive models, ReLU's hard cutoff (discontinuous derivative) creates 'blocky' optimization landscapes that can cause instability. SiLU is infinitely differentiable and 'smooth', leading to more stable updates. Furthermore, the non-monotonic 'dip' in SiLU allows for better generalization and prevents dead neurons.
</details>

### Q3: SwiGLU
How does SwiGLU differ from a standard Feed-Forward layer?
<details><summary>👉 Answer</summary>
Standard FFN: $f(Wx)W_2$. SwiGLU: $(\text{SiLU}(Wx) \otimes Vx)W_2$. SwiGLU uses a multiplicative 'gate' (the SiLU path) to dynamically filter information from the second path (V). It essentially learns to 'weight' the importance of every feature before passing it to the final projection.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner (Replicate)
1. Implement `LeakyReLU` from scratch using only NumPy and compare its derivative plot to standard `ReLU`.
2. Modify the benchmark code to include the `Tanh` activation and observe where its speed falls relative to ReLU and GELU.

### Tier 2: Intermediate (Extend)
1. **Mathematical Proof:** Using the chain rule, derive the derivative of `SiLU(x) = x * sigmoid(x)`. Show that it matches the formula in the table above.
2. **The Swish Beta:** Implement `Swish-Beta(x) = x * sigmoid(beta * x)`. Plot this for `beta = 0.1, 1.0, 10.0`. Explain why high beta makes it look more like ReLU.

### Tier 3: Advanced (Derive/Engineer)
1. **The Custom Kernel Challenge:** Modern activation functions are often slow because Python/PyTorch have to jump between C++ kernels for `linear` -> `activation` -> `multiply`. Research **Triton** or **TorchScript** and implement a fused `SwiGLU` kernel that combines the linear projections and the gating operation into a single CUDA call. Measure the memory bandwidth savings.
